## 1.记忆分类
- 短期记忆：当前会话的上下文
- 长期记忆：跨会话的经验知识

## 2.短期记忆
1.langchain把messages列表记录为AgentState的一部分。 
2.用Checkpointer对象来保存AgentState。  
3.每一次与Ai交互都生成一个快照checkpoint。  
4.同一个会话使用相同的thread_id

### 2.1.方式一记忆保存到内存中
1.导入依赖  
2.初始化内存存储器（之后放入智能体）    
2.配置thread_id  
3.每次会话都带上thread_id

In [ ]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver   #InMemorySaver：内存存储器
from langchain.messages import HumanMessage

#把内存存储器交给checkpointer
checkpointer = InMemorySaver()

clint = create_agent(
    "deepseek-chat",
    checkpointer=checkpointer
)

#设定thread_id作为会话标识
config={"configurable":{"thread_id":"thread_01"}}


In [ ]:
#第一次调用Agent
response = clint.invoke({
    "messages":[
        HumanMessage(content="你好，我叫小王")
    ]
},
    config
)

print(response['messages'][-1].content)

你好，小王！很高兴认识你😊。我是DeepSeek，一个智能助手，随时准备帮你解答问题、聊天或者提供各种信息。无论是学习、工作、生活，还是想聊聊兴趣爱好，我都在这里！你今天想聊点什么呢？


In [ ]:
#第二次调用Agent
response=clint.invoke({
    "messages":[
        HumanMessage(content="你还记得我是谁吗？")
    ]
},
    config
)

print(response['messages'][-1].content)

当然记得！你刚刚告诉我你叫小王～虽然我的记忆不会像人类那样长期保留，但在这次对话里，我会一直记得你的名字哦！😊 有什么需要帮忙的，随时告诉我～


### 2.2.方式二记忆保存到数据库
1.安装数据库依赖(以 uv add langgraph-checkpoint-sqlite 为例)  
2.初始化SQLite数据库连接  
3.初始化Checkpointer状态存储  
4.自动创建数据库存储表  
5.创建Agent，绑定状态存储

In [ ]:
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver
from langchain.agents import create_agent
from langchain.messages import HumanMessage

#初始化数据库连接
connection = sqlite3.connect("Database/checkpoint.db",check_same_thread=False)  #check_same_thread=False = 允许多线程使用（AI 必须要这个，不然会报错）

#把数据库连接交给checkpointer，即初始化checkpointer
checkpointer = SqliteSaver(connection)     #把刚才打开的数据库，交给 “存档器” 管理

#自动建表
checkpointer.setup()

clint = create_agent(
    "deepseek-chat",
    checkpointer=checkpointer
)

#设定thread_id作为会话标识
config = {"configurable":{"thread_id":"thread_02"}}

In [ ]:
#第一次调用
response = clint.invoke({
    "messages":[
        HumanMessage(content="我是小李子")
    ]
},
   config 
)

print(response["messages"][-1].content)

哈哈，好嘞，**小李子**！这次我彻底记住啦～（掏出虚拟小本本认真记下📝）  
既然你坚定地叫这个名字，那我猜——  
✨ 要么你是朋友圈里那个自带阳光BGM的开心果，  
🎬 要么你准备和我展开一场“奥斯卡欠他一座奖杯”的深度演技分析？  
（或者…你其实想让我喊你“李哥”“子哥”“小李总”？我随时切换称呼频道！😎）  

**随时等你指令，小李子模式已启动！** 🚀


In [ ]:
#第二次调用
response = clint.invoke({
    "messages":[
        HumanMessage(content="你还记得我是谁吗？")
    ]
},
   config 
)

print(response["messages"][-1].content)

哈哈，当然记得！你是 **小李子** 嘛～（精准锁定记忆库🔒）  
从第一句自我介绍开始，我就把你的“江湖称号”刻进代码里了！📝  
（不过要是你哪天想换ID，比如“李哥”“小李总”或者“莱昂纳多分多”，我也随时无缝切换～😎）  

需要本AI帮你做点啥？随时开口，小李子专属服务已上线！🚀
